# 04 — Entraînement modèle (Optuna × score_sortie + score_tele)

**Objectif :** entraîner le catalogue de modèles avec **Optuna** pour les deux scores, déployer les bundles dans `models/`.

> ⏱️ **Durée indicative : 30–60 minutes.** Laissez tourner sans interrompre le kernel.  
> Prérequis : parquets curated (`03_preparation_modele`) ou `python scripts/run_pipeline.py`.  
> Suite : `05_lancement_modele.ipynb`

Si des bundles récents existent déjà et que vous voulez **sauter** Optuna en séance, passez `FORCE_RETRAIN = False` ci-dessous.


## 1. Imports & chemins


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "patients.csv").is_file() and (p / "Sujet.md").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

from src.data.paths import ProjectPaths
from notebooks._utils.schema import FEATURES_CLES, SCHEMA_ATTENDU

paths = ProjectPaths(root=ROOT)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
print("Racine :", ROOT)
print("Curated :", paths.curated)
print("Models  :", paths.models)


## 2. Chargement des features curated


In [ ]:
p_so = paths.curated / "features_score_sortie.parquet"
p_te = paths.curated / "features_score_tele.parquet"
if not p_so.is_file() or not p_te.is_file():
    raise FileNotFoundError(
        f"Parquets manquants.\nAttendu :\n- {p_so}\n- {p_te}\n"
        "Exécutez 03_preparation_modele.ipynb ou : PYTHONPATH=. python scripts/run_pipeline.py"
    )

features_sortie = pd.read_parquet(p_so)
features_tele = pd.read_parquet(p_te)

for nom, df in [("sortie", features_sortie), ("tele", features_tele)]:
    manquantes = [c for c in FEATURES_CLES if c not in df.columns]
    if manquantes:
        raise AssertionError(f"{nom}: colonnes manquantes {manquantes}")
    print(f"✓ features_score_{nom}: {df.shape}")


## 3. Paramètres d’entraînement


In [ ]:
from src.model.deploy import train_score

# True = Optuna complet (spec jury). False = catalogue rapide (smoke).
USE_OPTUNA = True
OPTUNA_TRIALS = 25
# Si False et que les bundles existent, on réutilise sans ré-entraîner.
FORCE_RETRAIN = True

print("USE_OPTUNA =", USE_OPTUNA, "| trials =", OPTUNA_TRIALS, "| FORCE_RETRAIN =", FORCE_RETRAIN)


## 4. Entraînement score_sortie


In [ ]:
bundle_so = paths.models / "score_sortie_bundle.joblib"
metrics_so_path = paths.models / "score_sortie_metrics.json"

if not FORCE_RETRAIN and bundle_so.is_file() and metrics_so_path.is_file():
    import json
    metrics_sortie = json.loads(metrics_so_path.read_text(encoding="utf-8"))
    print("⏭ Bundle sortie déjà présent — skip (FORCE_RETRAIN=False)")
else:
    print("▶ train_score score_sortie (Optuna=%s)…" % USE_OPTUNA)
    metrics_sortie = train_score(
        features_sortie,
        score_name="score_sortie",
        paths=paths,
        use_optuna=USE_OPTUNA,
        optuna_trials=OPTUNA_TRIALS,
    )
    print("✓ score_sortie retenu :", metrics_sortie.get("retenu"))

display(pd.DataFrame([
    {"score": "sortie", "retenu": metrics_sortie.get("retenu"),
     "n_train": metrics_sortie.get("n_train"), "n_test": metrics_sortie.get("n_test"),
     "optuna": metrics_sortie.get("optuna")}
]))


## 5. Entraînement score_tele


In [ ]:
bundle_te = paths.models / "score_tele_bundle.joblib"
metrics_te_path = paths.models / "score_tele_metrics.json"

if not FORCE_RETRAIN and bundle_te.is_file() and metrics_te_path.is_file():
    import json
    metrics_tele = json.loads(metrics_te_path.read_text(encoding="utf-8"))
    print("⏭ Bundle télé déjà présent — skip (FORCE_RETRAIN=False)")
else:
    print("▶ train_score score_tele (Optuna=%s)…" % USE_OPTUNA)
    metrics_tele = train_score(
        features_tele,
        score_name="score_tele",
        paths=paths,
        use_optuna=USE_OPTUNA,
        optuna_trials=OPTUNA_TRIALS,
    )
    print("✓ score_tele retenu :", metrics_tele.get("retenu"))

display(pd.DataFrame([
    {"score": "tele", "retenu": metrics_tele.get("retenu"),
     "n_train": metrics_tele.get("n_train"), "n_test": metrics_tele.get("n_test"),
     "optuna": metrics_tele.get("optuna")}
]))


## 6. Tableau comparatif + artefacts artefacts


In [ ]:
import json

def resume_metrics(path: Path) -> dict:
    m = json.loads(path.read_text(encoding="utf-8"))
    retenu = m["retenu"]
    test = m.get(retenu, {}).get("test", {})
    return {
        "score": m.get("score"),
        "retenu": retenu,
        "pr_auc_test": test.get("pr_auc"),
        "roc_auc_test": test.get("roc_auc"),
        "f2_test": test.get("f2"),
        "seuil": test.get("threshold"),
        "optuna": m.get("optuna"),
    }

metrics_so_path = paths.models / "score_sortie_metrics.json"
metrics_te_path = paths.models / "score_tele_metrics.json"
comparatif = pd.DataFrame([resume_metrics(metrics_so_path), resume_metrics(metrics_te_path)])
display(comparatif)

print("Artefacts :")
for p in [
    paths.models / "score_sortie_bundle.joblib",
    paths.models / "score_sortie_metrics.json",
    paths.models / "score_tele_bundle.joblib",
    paths.models / "score_tele_metrics.json",
]:
    print(("✓" if p.is_file() else "✗"), p)

print("✓ Entraînement terminé — enchaînez avec 05_lancement_modele.ipynb")
